## Caso não tenha as libs instaladas no Kernel

In [1]:
# %pip install plotly pandas scikit-learn opencv-python
# %pip install --upgrade nbformat

## Import das libs

In [2]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import warnings
import json
from pathlib import Path

from IPython.display import Markdown, display
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

### Analise dos logs de voo em memmap

In [3]:
lista_dfs = []
caminhos_ficheiros = []


def listar_logs_voo_memmap():
    candidatos = sorted(Path("../logs").glob("voo_teste_*/manifest.json"))
    if not candidatos:
        candidatos = sorted(Path("logs").glob("voo_teste_*/manifest.json"))
    return candidatos

def carregar_log_voo_memmap(manifest_path):
    """Carrega intervalos de voo e reconstrui trajetoria e tempo relativos."""

    manifest_path = Path(manifest_path)

    try:
        texto = manifest_path.read_text(encoding="utf-8").strip()

        if not texto:
            print(f"Ignorando manifest vazio: {manifest_path}")
            return pd.DataFrame()

        manifest = json.loads(texto)

    except JSONDecodeError as e:
        print(f"Ignorando manifest invÃ¡lido: {manifest_path}")
        print(f"Erro: {e}")
        return pd.DataFrame()

    n = int(manifest.get("num_samples", 0))
    if n <= 0:
        return pd.DataFrame()

    intervalos_path = manifest_path.parent / manifest["arrays"]["flight_intervals"]

    if not intervalos_path.exists():
        print(f"Ignorando run sem memmap: {intervalos_path}")
        return pd.DataFrame()

    intervalos = np.load(intervalos_path, mmap_mode="r")[:n]

    df = pd.DataFrame(intervalos)
    df["run_id"] = manifest_path.parent.name
    df["manifest_path"] = str(manifest_path)

    df["tempo_s"] = df["dt_s"].fillna(0).cumsum()
    df["x_rel"] = df["delta_x_m"].fillna(0).cumsum()
    df["y_rel"] = df["delta_y_m"].fillna(0).cumsum()
    df["z_rel"] = df["delta_z_m"].fillna(0).cumsum()
    df["altitude_rel"] = -df["z_rel"]

    return df

for manifest_path in listar_logs_voo_memmap():
    df_temp = carregar_log_voo_memmap(manifest_path)
    if not df_temp.empty:
        caminhos_ficheiros.append(manifest_path)
        lista_dfs.append(df_temp)

if not lista_dfs:
    display(Markdown("Nenhum log novo em memmap encontrado em `logs/voo_teste_*/manifest.json`."))

### Deslocamento acumulado relativo

In [4]:
for caminho, df in zip(caminhos_ficheiros, lista_dfs):
    timestamp = caminho.parent.name.replace("voo_teste_", "")
    fig_3d = go.Figure()
    fig_3d.add_trace(go.Scatter3d(x=df["x_rel"], y=df["y_rel"], z=df["altitude_rel"], mode="lines", line=dict(color="royalblue", width=4), name="Deslocamento acumulado"))
    fig_3d.add_trace(go.Scatter3d(x=[0.0], y=[0.0], z=[0.0], mode="markers", marker=dict(color="green", size=6), name="Origem relativa"))
    fig_3d.add_trace(go.Scatter3d(x=[df["x_rel"].iloc[-1]], y=[df["y_rel"].iloc[-1]], z=[df["altitude_rel"].iloc[-1]], mode="markers", marker=dict(color="red", size=6, symbol="x"), name="Fim relativo"))
    fig_3d.update_layout(title=f"Deslocamento acumulado por deltas - Run: {timestamp}", scene=dict(xaxis_title="Delta X acumulado (m)", yaxis_title="Delta Y acumulado (m)", zaxis_title="Delta altitude acumulada (m)", camera=dict(eye=dict(x=1.5, y=1.5, z=0.5))), legend=dict(x=0, y=1), margin=dict(l=0, r=0, b=0, t=40))
    fig_3d.show()

### Variacao das velocidades angulares

In [5]:
for caminho, df in zip(caminhos_ficheiros, lista_dfs):
    timestamp = caminho.parent.name.replace("voo_teste_", "")
    fig_2d = go.Figure()
    fig_2d.add_trace(go.Scatter(x=df["tempo_s"], y=df["delta_roll_speed_rad_s"], mode="lines", name="Delta roll speed", opacity=0.7))
    fig_2d.add_trace(go.Scatter(x=df["tempo_s"], y=df["delta_pitch_speed_rad_s"], mode="lines", name="Delta pitch speed", opacity=0.7))
    fig_2d.add_trace(go.Scatter(x=df["tempo_s"], y=df["delta_yaw_speed_rad_s"], mode="lines", name="Delta yaw speed", opacity=0.7))
    fig_2d.update_layout(title=f"Variacao das velocidades angulares - Run: {timestamp}", xaxis_title="Tempo acumulado por intervalos (s)", yaxis_title="Delta velocidade angular (rad/s)", template="plotly_white", hovermode="x unified")
    fig_2d.show()

### Analise dos intervalos depth/flow em memmap

Esta analise usa as novas runs em `datasets/depth_ground_truth/run_*/manifest.json`. Cada amostra representa a variacao entre duas atualizacoes consecutivas da logica de proximidade visual por optical flow, a mesma logica que gera as flechas desenhadas na deteccao de obstaculos.

O foco deixa de ser o estado absoluto do drone em um frame isolado e passa a ser o deslocamento sincronizado do intervalo: deltas de posicao, atitude, IMU, comandos reativos, flow e depth ground truth. O objetivo e deixar os dados menos dependentes da origem da simulacao e mais genericos para treino e analise.

In [6]:
def localizar_raiz_projeto_memmap(nome_dataset="datasets"):
    """Localiza a raiz do projeto a partir do diretorio do notebook."""

    candidatos = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for candidato in candidatos:
        if (candidato / nome_dataset / "depth_ground_truth").exists():
            return candidato
    return Path.cwd()


def listar_runs_depth_memmap(base_dir):
    depth_dir = base_dir / "datasets" / "depth_ground_truth"
    return sorted(path.parent for path in depth_dir.glob("run_*/manifest.json"))


def carregar_manifesto_memmap(run_dir):
    with open(run_dir / "manifest.json", encoding="utf-8") as fp:
        return json.load(fp)


def abrir_array_memmap(run_dir, manifesto, chave):
    return np.load(run_dir / manifesto["arrays"][chave], mmap_mode="r")


def indices_intervalos_sincronizados(intervalos, max_age_s=0.08, max_interval_s=0.5):
    """Seleciona intervalos RGB/depth com tempos validos e sincronizados."""

    if intervalos.empty:
        return np.array([], dtype=int)
    dt_rgb = pd.to_numeric(intervalos["dt_s"], errors="coerce")
    dt_depth = pd.to_numeric(intervalos["depth_dt_s"], errors="coerce")
    depth_age = pd.to_numeric(intervalos["depth_age_s"], errors="coerce")
    mascara = (
        dt_rgb.notna() & dt_rgb.gt(0.0) & dt_rgb.le(max_interval_s)
        & dt_depth.notna() & dt_depth.gt(0.0) & dt_depth.le(max_interval_s)
        & depth_age.notna() & depth_age.le(max_age_s)
    )
    return np.flatnonzero(mascara.to_numpy())


def carregar_run_depth_memmap(run_dir):
    """Carrega arrays depth/flow e aplica o filtro de sincronizacao."""

    manifesto = carregar_manifesto_memmap(run_dir)
    n = int(manifesto.get("num_samples", 0))
    intervalos_mm = abrir_array_memmap(run_dir, manifesto, "intervals")
    intervalos_brutos = pd.DataFrame.from_records(intervalos_mm[:n]).copy()
    indices_validos = indices_intervalos_sincronizados(intervalos_brutos)
    intervalos = intervalos_brutos.iloc[indices_validos].reset_index(drop=True)
    if not intervalos.empty:
        intervalos["run_id"] = run_dir.name
        intervalos["ordem_intervalo"] = np.arange(len(intervalos))
    return {
        "run_dir": run_dir,
        "manifesto": manifesto,
        "intervalos": intervalos,
        "image_delta_bgr": abrir_array_memmap(run_dir, manifesto, "image_delta_bgr")[indices_validos],
        "depth_delta_log": abrir_array_memmap(run_dir, manifesto, "depth_delta_log")[indices_validos],
        "depth_delta_mask": abrir_array_memmap(run_dir, manifesto, "depth_delta_mask")[indices_validos],
        "flow_vectors": abrir_array_memmap(run_dir, manifesto, "flow_vectors")[indices_validos],
    }


def carregar_todas_runs_depth_memmap(base_dir):
    """Carrega todas as runs depth validas, isolando falhas por run."""

    runs = []
    for run_dir in listar_runs_depth_memmap(base_dir):
        try:
            run = carregar_run_depth_memmap(run_dir)
        except Exception as exc:
            print(f"Run ignorada em {run_dir.name}: {exc}")
            continue
        if len(run["intervalos"]) > 0:
            runs.append(run)
    return runs

In [7]:
raiz_projeto = localizar_raiz_projeto_memmap()
runs_depth_memmap = carregar_todas_runs_depth_memmap(raiz_projeto)

if not runs_depth_memmap:
    display(Markdown(
        "Nenhuma run nova em memmap encontrada em `datasets/depth_ground_truth/run_*/manifest.json`. "
        "Colete uma nova run com `save_ground_truth_dataset:=true`."
    ))
else:
    depth_run = runs_depth_memmap[-1]
    depth_df = depth_run["intervalos"].copy()
    manifesto = depth_run["manifesto"]
    print(f"Run analisada: {depth_run['run_dir'].name}")
    print(f"Intervalos depth/flow validos: {len(depth_df)}")
    print(f"Schema: {manifesto.get('schema_version')}")
    colunas_resumo = [
        "dt_s", "depth_age_s", "depth_dt_s", "delta_x_m", "delta_y_m", "delta_z_m",
        "delta_roll_rad", "delta_pitch_rad", "delta_yaw_heading_rad", "flow_valid_points",
        "flow_track_retention_pct", "flow_mag_p90_px", "radial_flow_p90_px",
        "delta_depth_p10_m", "delta_depth_p50_m", "delta_depth_close_5m_pp",
    ]
    display(depth_df[[c for c in colunas_resumo if c in depth_df.columns]].describe().T)
    fig_depth_delta = go.Figure()
    for coluna, nome in [("delta_depth_p10_m", "Delta depth P10"), ("delta_depth_p50_m", "Delta depth P50"), ("delta_depth_close_5m_pp", "Delta pixels < 5 m")]:
        if coluna in depth_df.columns:
            fig_depth_delta.add_trace(go.Scatter(x=depth_df["ordem_intervalo"], y=depth_df[coluna], mode="lines+markers", name=nome))
    fig_depth_delta.update_layout(title="Variacao de profundidade/proximidade por intervalo visual", xaxis_title="Intervalo visual em ordem de coleta", yaxis_title="Delta do intervalo", template="plotly_white", hovermode="x unified")
    fig_depth_delta.show()
    fig_flow_depth = px.scatter(depth_df, x="flow_mag_p90_px", y="delta_depth_close_5m_pp", color="radial_flow_p90_px", size="flow_valid_points", hover_data=["sample_id", "dt_s", "delta_x_m", "delta_yaw_heading_rad"], title="Flow visual x variacao de ocupacao proxima", template="plotly_white")
    fig_flow_depth.show()
    ranking = depth_df.assign(impacto_proximidade=depth_df["delta_depth_close_5m_pp"].abs()).sort_values(["impacto_proximidade", "flow_mag_p90_px"], ascending=False)
    display(Markdown("**Intervalos mais informativos para inspecao/treino:**"))
    display(ranking[["run_id", "sample_id", "dt_s", "flow_valid_points", "flow_mag_p90_px", "radial_flow_p90_px", "delta_depth_p10_m", "delta_depth_p50_m", "delta_depth_close_5m_pp", "delta_x_m", "delta_y_m", "delta_yaw_heading_rad"]].head(12))

Run analisada: run_20260723_235410
Intervalos depth/flow validos: 28
Schema: depth_interval_memmap_v1


,count,mean,std,min,25%,50%,75%,max
dt_s,28.0,0.155000,0.089088,0.064000,0.089000,0.132000,0.228000,0.460000
depth_age_s,28.0,0.043000,0.015724,0.032000,0.032000,0.032000,0.064000,0.068000
depth_dt_s,28.0,0.171571,0.096310,0.036000,0.100000,0.168000,0.232000,0.464000
delta_x_m,25.0,-0.049626,0.581432,-1.479023,-0.330044,-0.001095,0.152737,1.225008
delta_y_m,25.0,-0.048324,0.977393,-3.247337,-0.090393,0.255480,0.500622,1.325092
delta_z_m,25.0,-0.008000,0.036831,-0.160839,-0.005318,-0.000027,0.002975,0.030594
delta_roll_rad,28.0,-0.008476,0.086993,-0.204665,-0.019744,0.000057,0.025914,0.197799
delta_pitch_rad,28.0,0.004737,0.060414,-0.207659,-0.003144,0.000026,0.020605,0.112862
delta_yaw_heading_rad,28.0,0.054027,0.410406,-0.184119,-0.023698,-0.005933,0.000053,2.131047
flow_valid_points,28.0,79.071429,9.237318,61.000000,73.500000,80.000000,86.000000,105.000000


**Intervalos mais informativos para inspecao/treino:**

,run_id,sample_id,dt_s,flow_valid_points,flow_mag_p90_px,radial_flow_p90_px,delta_depth_p10_m,delta_depth_p50_m,delta_depth_close_5m_pp,delta_x_m,delta_y_m,delta_yaw_heading_rad
20,run_20260723_235410,23,0.132,66,139.385437,133.906464,-0.427684,-0.887846,8.885170,0.152737,0.334396,-0.184119
15,run_20260723_235410,18,0.164,91,35.275467,34.760674,-0.019692,-0.802399,7.276243,-0.634468,0.716576,-0.017890
12,run_20260723_235410,15,0.228,80,31.482992,30.705778,0.222095,0.987128,-4.356012,-1.479023,0.517473,-0.019398
8,run_20260723_235410,10,0.232,75,126.611038,121.881432,0.203139,0.557526,-3.846785,-0.115254,0.944674,0.038407
13,run_20260723_235410,16,0.228,86,85.273186,79.707092,-0.855775,0.859401,-3.686053,-0.514727,0.500622,-0.054446
9,run_20260723_235410,11,0.200,74,147.207291,53.279629,0.193637,0.519719,-3.453427,-0.106914,0.442521,0.062110
14,run_20260723_235410,17,0.064,69,95.282494,94.864967,-0.054937,-0.447659,3.200410,-0.144867,0.302681,-0.008209
26,run_20260723_235410,29,0.232,61,80.256851,79.574158,0.236623,0.006897,-2.779105,0.021051,-1.477131,-0.051588
25,run_20260723_235410,28,0.460,70,177.112900,148.771988,0.118497,0.310455,-2.589523,0.608585,-3.247337,-0.167320
22,run_20260723_235410,25,0.100,89,41.139503,39.425224,0.062822,-0.112233,1.705275,0.281551,-0.494843,0.025976


### Baseline MLP com vetores de variacao

A MLP agora trabalha com features tabulares de deslocamento entre intervalos visuais: deltas de estado, IMU, comandos, estatisticas do optical flow e estatisticas da diferenca de imagem estabilizada. Os alvos tambem sao deltas de depth/proximidade, e nao profundidades absolutas.

A comparacao contra `DummyRegressor(strategy="mean")` continua sendo usada como referencia minima: a MLP so e util se aprender uma relacao melhor que prever a variacao media observada no treino.

Para medir o efeito do aumento de dados sem alterar a dificuldade da avaliacao, as comparacoes cumulativas usam desde o primeiro marco as mesmas runs de validacao e teste. Apenas as runs de treino crescem entre os marcos. A validacao fixa e usada para decisoes de modelagem; o teste fixo serve somente para a comparacao final da curva de aprendizado.

A regressao principal usa duas camadas ocultas com dropout intercalado e early stopping. A melhor epoca e escolhida exclusivamente pela validacao fixa, e os pesos dessa epoca sao restaurados antes da avaliacao final no teste.

In [8]:
class DropoutMLP(nn.Module):
    """MLP com dropout entre as camadas ocultas."""

    def __init__(self, n_features, n_targets, hidden_layers=(64, 16), dropout=0.20):
        super().__init__()
        layers = []
        in_features = n_features
        for out_features in hidden_layers:
            layers.extend([
                nn.Linear(in_features, out_features),
                nn.ReLU(),
                nn.Dropout(dropout),
            ])
            in_features = out_features
        layers.append(nn.Linear(in_features, n_targets))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)


class EarlyStoppingDropoutRegressor:
    """Regressor PyTorch com normalizacao, dropout e early stopping por validacao."""

    def __init__(
        self,
        hidden_layers=(64, 16),
        dropout=0.20,
        learning_rate=1e-3,
        weight_decay=1e-4,
        batch_size=64,
        max_epochs=500,
        patience=30,
        min_delta=1e-4,
        random_state=42,
    ):
        self.hidden_layers = hidden_layers
        self.dropout = dropout
        self.learning_rate = learning_rate
        self.weight_decay = weight_decay
        self.batch_size = batch_size
        self.max_epochs = max_epochs
        self.patience = patience
        self.min_delta = min_delta
        self.random_state = random_state

    @staticmethod
    def _as_2d(values):
        array = np.asarray(values, dtype=np.float32)
        return array.reshape(-1, 1) if array.ndim == 1 else array

    @staticmethod
    def _mse(model, X_tensor, y_tensor):
        model.eval()
        with torch.no_grad():
            return float(nn.functional.mse_loss(model(X_tensor), y_tensor).item())

    def fit(self, X_train, y_train, X_val, y_val, X_test=None, y_test=None):
        """Treina e restaura os pesos da melhor epoca, escolhida somente pela validacao."""

        X_train = self._as_2d(X_train)
        X_val = self._as_2d(X_val)
        y_train = self._as_2d(y_train)
        y_val = self._as_2d(y_val)
        self.x_scaler_ = StandardScaler().fit(X_train)
        self.y_scaler_ = StandardScaler().fit(y_train)

        X_train_scaled = self.x_scaler_.transform(X_train).astype(np.float32)
        X_val_scaled = self.x_scaler_.transform(X_val).astype(np.float32)
        y_train_scaled = self.y_scaler_.transform(y_train).astype(np.float32)
        y_val_scaled = self.y_scaler_.transform(y_val).astype(np.float32)

        torch.manual_seed(self.random_state)
        np.random.seed(self.random_state)
        self.model_ = DropoutMLP(
            X_train_scaled.shape[1],
            y_train_scaled.shape[1],
            hidden_layers=self.hidden_layers,
            dropout=self.dropout,
        )
        optimizer = torch.optim.Adam(
            self.model_.parameters(),
            lr=self.learning_rate,
            weight_decay=self.weight_decay,
        )
        loss_fn = nn.MSELoss()
        generator = torch.Generator().manual_seed(self.random_state)
        train_loader = DataLoader(
            TensorDataset(
                torch.from_numpy(X_train_scaled),
                torch.from_numpy(y_train_scaled),
            ),
            batch_size=min(self.batch_size, len(X_train_scaled)),
            shuffle=True,
            generator=generator,
        )

        tensors = {
            "treino": (
                torch.from_numpy(X_train_scaled),
                torch.from_numpy(y_train_scaled),
            ),
            "validacao": (
                torch.from_numpy(X_val_scaled),
                torch.from_numpy(y_val_scaled),
            ),
        }
        if X_test is not None and y_test is not None:
            X_test_scaled = self.x_scaler_.transform(self._as_2d(X_test)).astype(np.float32)
            y_test_scaled = self.y_scaler_.transform(self._as_2d(y_test)).astype(np.float32)
            tensors["teste"] = (
                torch.from_numpy(X_test_scaled),
                torch.from_numpy(y_test_scaled),
            )

        best_loss = np.inf
        best_state = None
        epochs_without_improvement = 0
        history = []
        for epoch in range(1, self.max_epochs + 1):
            self.model_.train()
            for X_batch, y_batch in train_loader:
                optimizer.zero_grad()
                loss = loss_fn(self.model_(X_batch), y_batch)
                loss.backward()
                optimizer.step()

            epoch_losses = {
                split_name: self._mse(self.model_, *split_tensors)
                for split_name, split_tensors in tensors.items()
            }
            for split_name, mse in epoch_losses.items():
                history.append({
                    "epoca": epoch,
                    "split": split_name,
                    "mse_padronizado": mse,
                })

            validation_loss = epoch_losses["validacao"]
            if validation_loss < best_loss - self.min_delta:
                best_loss = validation_loss
                best_state = {
                    name: value.detach().cpu().clone()
                    for name, value in self.model_.state_dict().items()
                }
                self.best_epoch_ = epoch
                epochs_without_improvement = 0
            else:
                epochs_without_improvement += 1
                if epochs_without_improvement >= self.patience:
                    break

        self.model_.load_state_dict(best_state)
        self.best_validation_loss_ = best_loss
        self.stopped_epoch_ = epoch
        self.history_ = pd.DataFrame(history)
        self.n_outputs_ = y_train.shape[1]
        return self

    def predict(self, X):
        X_scaled = self.x_scaler_.transform(self._as_2d(X)).astype(np.float32)
        self.model_.eval()
        with torch.no_grad():
            pred_scaled = self.model_(torch.from_numpy(X_scaled)).numpy()
        prediction = self.y_scaler_.inverse_transform(pred_scaled)
        return prediction.ravel() if self.n_outputs_ == 1 else prediction


TARGETS_MLP_DELTA = ["delta_depth_p10_m", "delta_depth_p50_m", "delta_depth_p90_m", "delta_depth_close_2m_pp", "delta_depth_close_5m_pp", "delta_depth_close_10m_pp"]
REGRESSOR_CONFIG = {
    "hidden_layers": (64, 16),
    "dropout": 0.30,
    "max_epochs": 300,
    "patience": 50,
    "min_delta": 1e-4,
    "random_state": 42,
}


def features_image_delta(img):
    arr = np.asarray(img, dtype=np.float32)
    abs_arr = np.abs(arr)
    return {"img_delta_mean": float(arr.mean()), "img_delta_std": float(arr.std()), "img_delta_abs_mean": float(abs_arr.mean()), "img_delta_abs_p90": float(np.percentile(abs_arr, 90))}


def features_flow_vectors(vetores, valid_points):
    n = int(max(0, min(valid_points, len(vetores))))
    if n == 0:
        return {"flow_vec_rel_std": 0.0, "flow_vec_xy_mean": 0.0, "flow_vec_radial_mean": 0.0, "flow_vec_risk_mean": 0.0}
    v = np.asarray(vetores[:n], dtype=np.float32)
    return {"flow_vec_rel_std": float(v[:, :2].std()), "flow_vec_xy_mean": float(v[:, 2:4].mean()), "flow_vec_radial_mean": float(v[:, 4].mean()), "flow_vec_risk_mean": float(v[:, 5].mean())}


def montar_dataset_mlp_delta_depth(base_dir):
    """Monta features, alvos e metadados dos intervalos depth/flow validos."""

    linhas_x, linhas_y, linhas_meta = [], [], []
    for run in carregar_todas_runs_depth_memmap(base_dir):
        df = run["intervalos"].reset_index(drop=True)
        for i, row in df.iterrows():
            if any(col not in row.index or not np.isfinite(row[col]) for col in TARGETS_MLP_DELTA):
                continue
            feats = {}
            for coluna in df.select_dtypes(include=[np.number]).columns:
                if coluna in TARGETS_MLP_DELTA or coluna.startswith("delta_depth_") or coluna in {"delta_valid_px_pct", "sample_id", "ordem_intervalo"}:
                    continue
                valor = row[coluna]
                feats[coluna] = 0.0 if pd.isna(valor) else float(valor)
            feats.update(features_image_delta(run["image_delta_bgr"][i]))
            feats.update(features_flow_vectors(run["flow_vectors"][i], row.get("flow_valid_points", 0)))
            linhas_x.append(feats)
            linhas_y.append({alvo: float(row[alvo]) for alvo in TARGETS_MLP_DELTA})
            linhas_meta.append({"run_id": run["run_dir"].name, "sample_id": int(row.get("sample_id", i + 1)), "ordem_intervalo": int(row.get("ordem_intervalo", i))})
    return pd.DataFrame(linhas_x), pd.DataFrame(linhas_y), pd.DataFrame(linhas_meta)


RUNS_VALIDACAO_FIXAS = {"run_20260716_221545", "run_20260715_215803"}
RUNS_TESTE_FIXAS = {"run_20260715_220346", "run_20260715_220437", "run_20260716_231519"}
MARCOS_BASE_COMPARACAO = (9, 17, 25, 33)

def marcos_comparacao(total_runs):
    """Retorna os marcos historicos e inclui sempre o total atual de runs."""

    marcos = [marco for marco in MARCOS_BASE_COMPARACAO if marco <= total_runs]
    if total_runs and total_runs not in marcos:
        marcos.append(total_runs)
    return tuple(marcos)


def recortar_marco_runs(X, y, meta_df, runs_ordenadas, marco_runs):
    """Recorta X, y e metadados para um marco cumulativo de runs."""

    runs_marco = set(runs_ordenadas[:marco_runs])
    mascara = meta_df["run_id"].isin(runs_marco).to_numpy()
    return tuple(df.loc[mascara].reset_index(drop=True) for df in (X, y, meta_df))


def separar_intervalos(meta_df):
    """Separa intervalos por runs fixas, com fallback temporal para datasets antigos."""

    n = len(meta_df)
    indices = np.arange(n)
    runs = set(meta_df["run_id"].dropna().unique()) if n else set()
    runs_fixas = RUNS_VALIDACAO_FIXAS | RUNS_TESTE_FIXAS
    if runs_fixas.issubset(runs):
        train_runs = runs - runs_fixas
        if not train_runs:
            raise ValueError("Nenhuma run restante para treino apos aplicar os splits fixos.")
        return {
            "train": indices[meta_df["run_id"].isin(train_runs).to_numpy()],
            "val": indices[meta_df["run_id"].isin(RUNS_VALIDACAO_FIXAS).to_numpy()],
            "test": indices[meta_df["run_id"].isin(RUNS_TESTE_FIXAS).to_numpy()],
            "modo": "por run_id com validacao/teste fixos",
            "train_runs": sorted(train_runs),
            "val_runs": sorted(RUNS_VALIDACAO_FIXAS),
            "test_runs": sorted(RUNS_TESTE_FIXAS),
        }
    n_train = max(1, int(n * 0.70))
    n_val = max(1, int(n * 0.15))
    return {
        "train": indices[:n_train],
        "val": indices[n_train:n_train + n_val],
        "test": indices[n_train + n_val:],
        "modo": "temporal por intervalos",
    }


def avaliar_delta(y_true, y_pred, targets, modelo, split):
    """Calcula MAE, RMSE e correlacao de cada alvo de profundidade."""

    linhas = []
    for i, alvo in enumerate(targets):
        real = np.asarray(y_true[:, i], dtype=float)
        pred = np.asarray(y_pred[:, i], dtype=float)
        corr = float(np.corrcoef(real, pred)[0, 1]) if len(real) > 1 and np.std(real) > 1e-9 and np.std(pred) > 1e-9 else np.nan
        linhas.append({"modelo": modelo, "split": split, "alvo_delta": alvo, "MAE": float(mean_absolute_error(real, pred)), "RMSE": float(mean_squared_error(real, pred) ** 0.5), "corr": corr})
    return linhas


def treinar_avaliar_mlp_delta_depth(X, y, meta_df):
    """Treina a MLP com dropout/early stopping e o baseline nos mesmos splits."""

    targets = list(y.columns)
    split = separar_intervalos(meta_df)
    if len(split["test"]) == 0:
        split["test"] = split["val"]
    if len(split["val"]) == 0:
        split["val"] = split["test"]
    Xv = X.replace([np.inf, -np.inf], np.nan).fillna(0.0).to_numpy(float)
    yv = y.replace([np.inf, -np.inf], np.nan).fillna(0.0).to_numpy(float)
    mlp = EarlyStoppingDropoutRegressor(**REGRESSOR_CONFIG)
    mlp.fit(
        Xv[split["train"]],
        yv[split["train"]],
        Xv[split["val"]],
        yv[split["val"]],
        Xv[split["test"]],
        yv[split["test"]],
    )
    dummy = DummyRegressor(strategy="mean")
    dummy.fit(Xv[split["train"]], yv[split["train"]])
    linhas, predicoes = [], {}
    for nome_split, idx in [("treino", split["train"]), ("validacao", split["val"]), ("teste", split["test"])]:
        pred_mlp = mlp.predict(Xv[idx])
        pred_dummy = dummy.predict(Xv[idx])
        predicoes[nome_split] = {"idx": idx, "real": yv[idx], "mlp": pred_mlp, "dummy": pred_dummy}
        linhas.extend(avaliar_delta(yv[idx], pred_mlp, targets, "MLP", nome_split))
        linhas.extend(avaliar_delta(yv[idx], pred_dummy, targets, "Media treino", nome_split))
    return mlp, dummy, split, pd.DataFrame(linhas), predicoes


raiz_mlp = localizar_raiz_projeto_memmap()
X_mlp, y_mlp, meta_mlp = montar_dataset_mlp_delta_depth(raiz_mlp)
if len(X_mlp) < 10:
    display(Markdown("Dataset insuficiente para treinar a MLP de deltas. Colete mais intervalos depth/flow em memmap."))
else:
    runs_ordenadas_mlp = list(meta_mlp["run_id"].drop_duplicates())
    resultados_por_volume = []
    artefatos_por_volume = {}
    for marco_runs in marcos_comparacao(len(runs_ordenadas_mlp)):
        if len(runs_ordenadas_mlp) < marco_runs:
            continue
        X_marco, y_marco, meta_marco = recortar_marco_runs(
            X_mlp, y_mlp, meta_mlp, runs_ordenadas_mlp, marco_runs
        )
        modelo, baseline, split, resultados, predicoes = treinar_avaliar_mlp_delta_depth(X_marco, y_marco, meta_marco)
        resultados = resultados.assign(
            marco_runs=marco_runs,
            estrategia_split="validacao_teste_fixos",
        )
        resultados_por_volume.append(resultados)
        artefatos_por_volume[marco_runs] = {"modelo": modelo, "baseline": baseline, "split": split, "predicoes": predicoes, "meta": meta_marco}

    resultados_mlp_cumulativos = pd.concat(resultados_por_volume, ignore_index=True)
    caminho_resultados_mlp = raiz_mlp / "estudos_e_analises" / "comparacao_mlp_splits_fixos.csv"
    resultados_mlp_cumulativos.to_csv(caminho_resultados_mlp, index=False)
    marco_atual = max(artefatos_por_volume)
    artefato_atual = artefatos_por_volume[marco_atual]
    modelo_mlp_delta = artefato_atual["modelo"]
    baseline_media_delta = artefato_atual["baseline"]
    split_mlp = artefato_atual["split"]
    predicoes_mlp_delta = artefato_atual["predicoes"]
    resultados_mlp_delta = resultados_mlp_cumulativos[resultados_mlp_cumulativos["marco_runs"] == marco_atual].drop(columns="marco_runs")

    display(Markdown(
        f"**Comparacao MLP com splits fixos:** validacao={sorted(RUNS_VALIDACAO_FIXAS)}; "
        f"teste={sorted(RUNS_TESTE_FIXAS)}. Apenas o treino cresce entre "
        f"{list(artefatos_por_volume)} runs."
    ))
    resumo_splits = pd.DataFrame([
        {
            "marco_runs": marco,
            "intervalos": len(artefatos_por_volume[marco]["meta"]),
            "treino": len(artefatos_por_volume[marco]["split"]["train"]),
            "validacao": len(artefatos_por_volume[marco]["split"]["val"]),
            "teste": len(artefatos_por_volume[marco]["split"]["test"]),
            "treino_pct": len(artefatos_por_volume[marco]["split"]["train"]) / len(artefatos_por_volume[marco]["meta"]) * 100.0,
            "validacao_pct": len(artefatos_por_volume[marco]["split"]["val"]) / len(artefatos_por_volume[marco]["meta"]) * 100.0,
            "teste_pct": len(artefatos_por_volume[marco]["split"]["test"]) / len(artefatos_por_volume[marco]["meta"]) * 100.0,
        }
        for marco in artefatos_por_volume
    ])
    display(resumo_splits)
    resultados_teste_cumulativos = resultados_mlp_cumulativos[resultados_mlp_cumulativos["split"] == "teste"]
    display(resultados_teste_cumulativos.sort_values(["alvo_delta", "marco_runs", "modelo"]).reset_index(drop=True))
    px.line(resultados_teste_cumulativos, x="marco_runs", y="MAE", color="modelo", facet_col="alvo_delta", facet_col_wrap=3, markers=True, title=f"Curva de aprendizado no teste fixo: {list(artefatos_por_volume)} runs", template="plotly_white").show()

    loss_por_volume = resultados_mlp_cumulativos.query("modelo == 'MLP'").copy()
    loss_por_volume["MSE"] = loss_por_volume["RMSE"] ** 2
    px.line(
        loss_por_volume,
        x="marco_runs",
        y="MSE",
        color="split",
        facet_col="alvo_delta",
        facet_col_wrap=3,
        markers=True,
        title="Loss da MLP principal por volume: treino crescente, validacao e teste fixos",
        labels={"marco_runs": "Runs disponiveis", "MSE": "MSE", "split": "Conjunto"},
        template="plotly_white",
    ).show()

    X_atual, y_atual, meta_atual = recortar_marco_runs(
        X_mlp, y_mlp, meta_mlp, runs_ordenadas_mlp, marco_atual
    )
    curvas_convergencia = modelo_mlp_delta.history_.copy()
    caminho_curvas_loss = raiz_mlp / "estudos_e_analises" / "curvas_loss_mlp.csv"
    curvas_convergencia.assign(marco_runs=marco_atual).to_csv(caminho_curvas_loss, index=False)
    validacao_loss = curvas_convergencia.query("split == 'validacao'")
    melhor_linha = validacao_loss.loc[validacao_loss["mse_padronizado"].idxmin()]
    melhor_epoca = modelo_mlp_delta.best_epoch_
    fig_loss_epocas = px.line(
        curvas_convergencia,
        x="epoca",
        y="mse_padronizado",
        color="split",
        title=f"MLP com dropout e early stopping ({marco_atual} runs)",
        labels={"epoca": "Epoca", "mse_padronizado": "MSE padronizado", "split": "Conjunto"},
        template="plotly_white",
    )
    fig_loss_epocas.add_vline(
        x=melhor_epoca,
        line_dash="dash",
        annotation_text=f"early stopping: melhor epoca {melhor_epoca}",
    )
    fig_loss_epocas.show()

    loss_final = curvas_convergencia.groupby("split", as_index=False).tail(1)
    display(pd.DataFrame([{
        "marco_runs": marco_atual,
        "melhor_epoca_validacao": melhor_epoca,
        "menor_loss_validacao": float(melhor_linha["mse_padronizado"]),
        "loss_treino_final": float(loss_final.query("split == 'treino'")["mse_padronizado"].iloc[0]),
        "loss_validacao_final": float(loss_final.query("split == 'validacao'")["mse_padronizado"].iloc[0]),
        "loss_teste_final": float(loss_final.query("split == 'teste'")["mse_padronizado"].iloc[0]),
        "epoca_interrupcao": modelo_mlp_delta.stopped_epoch_,
        "dropout": modelo_mlp_delta.dropout,
        "patience": modelo_mlp_delta.patience,
    }]))

    alvo_plot = "delta_depth_close_5m_pp" if "delta_depth_close_5m_pp" in y_mlp.columns else y_mlp.columns[0]
    alvo_idx = list(y_mlp.columns).index(alvo_plot)
    pred_teste = predicoes_mlp_delta["teste"]
    meta_teste = artefato_atual["meta"].iloc[pred_teste["idx"]].reset_index(drop=True)
    fig_pred = go.Figure()
    fig_pred.add_trace(go.Scatter(x=meta_teste.index, y=pred_teste["real"][:, alvo_idx], mode="lines+markers", name="real"))
    fig_pred.add_trace(go.Scatter(x=meta_teste.index, y=pred_teste["mlp"][:, alvo_idx], mode="lines+markers", name="MLP"))
    fig_pred.add_trace(go.Scatter(x=meta_teste.index, y=pred_teste["dummy"][:, alvo_idx], mode="lines", name="media treino"))
    fig_pred.update_layout(title=f"Predicao no teste fixo para {alvo_plot} com {marco_atual} runs", xaxis_title="Intervalos de teste em ordem temporal", yaxis_title="Delta do alvo", template="plotly_white", hovermode="x unified"); fig_pred.show()
    display(Markdown("**Intervalos do teste fixo para inspecao:**"))
    display(pd.DataFrame({"run_id": meta_teste["run_id"], "sample_id": meta_teste["sample_id"], f"{alvo_plot}_real": pred_teste["real"][:, alvo_idx], f"{alvo_plot}_mlp": pred_teste["mlp"][:, alvo_idx], f"{alvo_plot}_baseline_media": pred_teste["dummy"][:, alvo_idx]}).head(15))

**Comparacao MLP com splits fixos:** validacao=['run_20260715_215803', 'run_20260716_221545']; teste=['run_20260715_220346', 'run_20260715_220437', 'run_20260716_231519']. Apenas o treino cresce entre [9, 17, 25, 33, 40] runs.

,marco_runs,intervalos,treino,validacao,teste,treino_pct,validacao_pct,teste_pct
0,9,323,144,85,94,44.582043,26.315789,29.102167
1,17,723,544,85,94,75.242047,11.756570,13.001383
2,25,827,648,85,94,78.355502,10.278114,11.366385
3,33,1015,836,85,94,82.364532,8.374384,9.261084
4,40,1234,1055,85,94,85.494327,6.888169,7.617504


,modelo,split,alvo_delta,MAE,RMSE,corr,marco_runs,estrategia_split
0,MLP,teste,delta_depth_close_10m_pp,0.938076,1.270671,0.098302,9,validacao_teste_fixos
1,Media treino,teste,delta_depth_close_10m_pp,0.922800,1.241606,NaN,9,validacao_teste_fixos
2,MLP,teste,delta_depth_close_10m_pp,0.831542,1.167291,0.328230,17,validacao_teste_fixos
3,Media treino,teste,delta_depth_close_10m_pp,0.926351,1.244116,NaN,17,validacao_teste_fixos
4,MLP,teste,delta_depth_close_10m_pp,0.857675,1.173298,0.328743,25,validacao_teste_fixos
5,Media treino,teste,delta_depth_close_10m_pp,0.925334,1.243389,NaN,25,validacao_teste_fixos
6,MLP,teste,delta_depth_close_10m_pp,0.852259,1.169102,0.305074,33,validacao_teste_fixos
7,Media treino,teste,delta_depth_close_10m_pp,0.914198,1.235791,NaN,33,validacao_teste_fixos
8,MLP,teste,delta_depth_close_10m_pp,0.840096,1.162781,0.323922,40,validacao_teste_fixos
9,Media treino,teste,delta_depth_close_10m_pp,0.909589,1.232640,NaN,40,validacao_teste_fixos


,marco_runs,melhor_epoca_validacao,menor_loss_validacao,loss_treino_final,loss_validacao_final,loss_teste_final,epoca_interrupcao,dropout,patience
0,40,17,0.539201,0.643313,0.552652,0.585289,67,0.3,50


**Intervalos do teste fixo para inspecao:**

,run_id,sample_id,delta_depth_close_5m_pp_real,delta_depth_close_5m_pp_mlp,delta_depth_close_5m_pp_baseline_media
0,run_20260715_220346,1,0.000000,-0.034823,-0.096808
1,run_20260715_220346,2,-0.000229,-0.029874,-0.096808
2,run_20260715_220346,3,0.000000,-0.003463,-0.096808
3,run_20260715_220346,4,-0.844081,-0.816960,-0.096808
4,run_20260715_220346,5,0.528050,0.705478,-0.096808
5,run_20260715_220346,6,-0.706924,-0.326386,-0.096808
6,run_20260715_220346,7,-3.853416,-0.572371,-0.096808
7,run_20260715_220346,8,1.590667,0.328564,-0.096808
8,run_20260715_220346,9,0.498529,0.322313,-0.096808
9,run_20260715_220346,10,-0.485029,0.329712,-0.096808


### Explicabilidade pelo gradiente do erro nas entradas

Esta etapa usa somente o melhor modelo ja treinado e restaurado pelo early stopping. Nenhuma feature ou run e removida e a rede nao e retreinada.

Para cada alvo, o erro quadratico padronizado e retropropagado ate as entradas. O modulo do gradiente indica quanto uma pequena mudanca em cada feature afetaria o erro daquele alvo. Como as entradas foram padronizadas com os parametros do treino, as sensibilidades ficam comparaveis. As importancias sao normalizadas para somar 100% em cada alvo.

A analise principal usa apenas a validacao fixa. O teste permanece reservado para a avaliacao final e nao participa da escolha de features, transformacoes ou arquitetura.


In [9]:
def classificar_grupo_feature(nome):
    """Agrupa features apenas para resumir as atribuicoes individuais."""

    nome = nome.lower()
    if nome.startswith("img_"):
        return "Diferenca de imagem"
    if any(token in nome for token in ("avoid_", "obstacle_", "pan_comp")):
        return "Comandos reativos"
    if "flow" in nome or "point_risk" in nome:
        return "Optical flow e risco"
    if any(token in nome for token in ("gyro", "accel", "roll", "pitch", "yaw")):
        return "IMU e atitude"
    if any(token in nome for token in ("delta_x", "delta_y", "delta_z", "smooth_v", "velocity")):
        return "Movimento"
    if any(token in nome for token in ("dt_s", "depth_age", "depth_dt", "timestamp")):
        return "Sincronizacao"
    return "Outras"


def calcular_importancia_gradiente_erro(modelo, X, y, nomes_features, nomes_alvos):
    """Calcula |d erro / d entrada| no conjunto informado sem retreinar a rede."""

    Xv = (
        X.replace([np.inf, -np.inf], np.nan)
        .fillna(0.0)
        .to_numpy(dtype=np.float32)
    )
    yv = (
        y.replace([np.inf, -np.inf], np.nan)
        .fillna(0.0)
        .to_numpy(dtype=np.float32)
    )
    X_scaled = modelo.x_scaler_.transform(Xv).astype(np.float32)
    y_scaled = modelo.y_scaler_.transform(yv).astype(np.float32)
    X_tensor = torch.tensor(X_scaled, dtype=torch.float32, requires_grad=True)
    y_tensor = torch.tensor(y_scaled, dtype=torch.float32)

    rede = modelo.model_
    rede.eval()
    with torch.no_grad():
        pred_scaled = rede(X_tensor.detach()).cpu().numpy()
    pred_original = modelo.y_scaler_.inverse_transform(pred_scaled)

    linhas_globais = []
    atribuicoes_locais = {}
    for alvo_idx, alvo in enumerate(nomes_alvos):
        rede.zero_grad(set_to_none=True)
        if X_tensor.grad is not None:
            X_tensor.grad.zero_()

        pred = rede(X_tensor)
        erros_amostra = (pred[:, alvo_idx] - y_tensor[:, alvo_idx]).square()
        erros_amostra.mean().backward()
        gradientes_abs = X_tensor.grad.detach().abs().cpu().numpy().copy()

        importancia_global = gradientes_abs.mean(axis=0)
        importancia_global /= max(float(importancia_global.sum()), 1e-12)
        soma_local = gradientes_abs.sum(axis=1, keepdims=True)
        importancia_local = gradientes_abs / np.clip(soma_local, 1e-12, None)
        atribuicoes_locais[alvo] = importancia_local

        for feature, importancia in zip(nomes_features, importancia_global):
            linhas_globais.append({
                "alvo_delta": alvo,
                "feature": feature,
                "grupo": classificar_grupo_feature(feature),
                "importancia": float(importancia),
                "importancia_pct": float(importancia * 100.0),
            })

    importancia_df = pd.DataFrame(linhas_globais)
    importancia_df["ranking_no_alvo"] = (
        importancia_df.groupby("alvo_delta")["importancia"]
        .rank(method="first", ascending=False)
        .astype(int)
    )
    return importancia_df, atribuicoes_locais, pred_original


if "modelo_mlp_delta" not in globals() or "X_atual" not in globals():
    display(Markdown(
        "Execute primeiro a celula de treinamento da MLP para gerar o melhor modelo."
    ))
else:
    indices_validacao = split_mlp["val"]
    X_explicacao = X_atual.iloc[indices_validacao].reset_index(drop=True)
    y_explicacao = y_atual.iloc[indices_validacao].reset_index(drop=True)
    meta_explicacao = meta_atual.iloc[indices_validacao].reset_index(drop=True)
    nomes_features = list(X_explicacao.columns)
    nomes_alvos = list(y_explicacao.columns)

    importancia_gradiente_df, importancia_local_por_alvo, pred_explicacao = (
        calcular_importancia_gradiente_erro(
            modelo_mlp_delta,
            X_explicacao,
            y_explicacao,
            nomes_features,
            nomes_alvos,
        )
    )
    caminho_importancia = (
        raiz_mlp
        / "estudos_e_analises"
        / "importancia_gradiente_erro_validacao.csv"
    )
    importancia_gradiente_df.to_csv(caminho_importancia, index=False)

    display(Markdown(
        f"**Modelo explicado:** pesos da epoca {modelo_mlp_delta.best_epoch_}, "
        f"dropout desativado durante a explicacao (`eval`) e "
        f"{len(X_explicacao)} intervalos da validacao fixa. "
        "O conjunto de teste nao foi utilizado."
    ))

    top_features_por_alvo = (
        importancia_gradiente_df
        .sort_values(["alvo_delta", "importancia"], ascending=[True, False])
        .groupby("alvo_delta", as_index=False)
        .head(8)
    )
    display(Markdown("**Entradas mais associadas ao erro de cada alvo:**"))
    display(
        top_features_por_alvo[
            ["alvo_delta", "ranking_no_alvo", "feature", "grupo", "importancia_pct"]
        ].round({"importancia_pct": 2})
    )

    features_heatmap = (
        importancia_gradiente_df
        .groupby("feature")["importancia"]
        .mean()
        .nlargest(20)
        .index
    )
    matriz_importancia = (
        importancia_gradiente_df[
            importancia_gradiente_df["feature"].isin(features_heatmap)
        ]
        .pivot(index="alvo_delta", columns="feature", values="importancia_pct")
        .reindex(index=nomes_alvos, columns=features_heatmap)
    )
    fig_importancia = px.imshow(
        matriz_importancia,
        aspect="auto",
        color_continuous_scale="Blues",
        text_auto=".1f",
        labels={
            "x": "Feature de entrada",
            "y": "Alvo",
            "color": "Importancia (%)",
        },
        title="Importancia pelo gradiente do erro: alvo x entrada",
    )
    fig_importancia.update_xaxes(tickangle=45)
    fig_importancia.show()

    importancia_grupos_df = (
        importancia_gradiente_df
        .groupby(["alvo_delta", "grupo"], as_index=False)["importancia_pct"]
        .sum()
    )
    fig_grupos = px.bar(
        importancia_grupos_df,
        x="grupo",
        y="importancia_pct",
        color="grupo",
        facet_col="alvo_delta",
        facet_col_wrap=3,
        title="Soma das importancias individuais por grupo de entrada",
        labels={
            "grupo": "Grupo",
            "importancia_pct": "Importancia acumulada (%)",
            "alvo_delta": "Alvo",
        },
        template="plotly_white",
    )
    fig_grupos.update_xaxes(tickangle=35)
    fig_grupos.show()

    alvo_detalhe = (
        "delta_depth_close_5m_pp"
        if "delta_depth_close_5m_pp" in nomes_alvos
        else nomes_alvos[0]
    )
    alvo_idx = nomes_alvos.index(alvo_detalhe)
    real_detalhe = y_explicacao[alvo_detalhe].to_numpy(dtype=float)
    pred_detalhe = pred_explicacao[:, alvo_idx]
    erro_abs = np.abs(pred_detalhe - real_detalhe)
    quantidade_extremos = min(12, len(erro_abs))
    indices_extremos = np.argsort(erro_abs)[-quantidade_extremos:][::-1]

    features_detalhe = (
        importancia_gradiente_df
        .query("alvo_delta == @alvo_detalhe")
        .nlargest(15, "importancia")["feature"]
        .tolist()
    )
    indices_features = [nomes_features.index(feature) for feature in features_detalhe]
    matriz_extremos = (
        importancia_local_por_alvo[alvo_detalhe][indices_extremos][:, indices_features]
        * 100.0
    )
    rotulos_extremos = [
        (
            f"{meta_explicacao.iloc[idx]['run_id']} / "
            f"{int(meta_explicacao.iloc[idx]['sample_id'])} "
            f"(erro={erro_abs[idx]:.2f})"
        )
        for idx in indices_extremos
    ]
    fig_extremos = go.Figure(go.Heatmap(
        z=matriz_extremos,
        x=features_detalhe,
        y=rotulos_extremos,
        colorscale="Reds",
        colorbar={"title": "Importancia local (%)"},
    ))
    fig_extremos.update_layout(
        title=f"Quais entradas influenciaram os maiores erros em {alvo_detalhe}",
        xaxis_title="Feature",
        yaxis_title="Intervalo de validacao",
        template="plotly_white",
    )
    fig_extremos.update_xaxes(tickangle=45)
    fig_extremos.show()

    linhas_extremos = []
    importancia_local = importancia_local_por_alvo[alvo_detalhe]
    for idx in indices_extremos:
        top_indices = np.argsort(importancia_local[idx])[-3:][::-1]
        linhas_extremos.append({
            "run_id": meta_explicacao.iloc[idx]["run_id"],
            "sample_id": int(meta_explicacao.iloc[idx]["sample_id"]),
            "real": float(real_detalhe[idx]),
            "predito": float(pred_detalhe[idx]),
            "erro_absoluto": float(erro_abs[idx]),
            "feature_1": nomes_features[top_indices[0]],
            "importancia_1_pct": float(importancia_local[idx, top_indices[0]] * 100.0),
            "feature_2": nomes_features[top_indices[1]],
            "importancia_2_pct": float(importancia_local[idx, top_indices[1]] * 100.0),
            "feature_3": nomes_features[top_indices[2]],
            "importancia_3_pct": float(importancia_local[idx, top_indices[2]] * 100.0),
        })
    extremos_gradiente_df = pd.DataFrame(linhas_extremos)
    caminho_extremos = (
        raiz_mlp
        / "estudos_e_analises"
        / "maiores_erros_gradiente_validacao.csv"
    )
    extremos_gradiente_df.to_csv(caminho_extremos, index=False)
    display(Markdown(
        f"**Maiores erros de validacao em `{alvo_detalhe}` e suas tres entradas "
        "mais influentes:**"
    ))
    display(extremos_gradiente_df.round(2))


**Modelo explicado:** pesos da epoca 17, dropout desativado durante a explicacao (`eval`) e 85 intervalos da validacao fixa. O conjunto de teste nao foi utilizado.

**Entradas mais associadas ao erro de cada alvo:**

,alvo_delta,ranking_no_alvo,feature,grupo,importancia_pct
234,delta_depth_close_10m_pp,1,gyro_y_integral_rad,IMU e atitude,11.83
227,delta_depth_close_10m_pp,2,delta_pitch_rad,IMU e atitude,10.30
225,delta_depth_close_10m_pp,3,delta_z_m,Movimento,6.73
231,delta_depth_close_10m_pp,4,delta_gyro_y_rad_s,IMU e atitude,4.57
236,delta_depth_close_10m_pp,5,delta_accel_x_m_s2,IMU e atitude,4.12
251,delta_depth_close_10m_pp,6,flow_mag_p90_px,Optical flow e risco,3.63
237,delta_depth_close_10m_pp,7,delta_accel_y_m_s2,IMU e atitude,2.89
233,delta_depth_close_10m_pp,8,gyro_x_integral_rad,IMU e atitude,2.84
146,delta_depth_close_2m_pp,1,gyro_y_integral_rad,IMU e atitude,12.82
139,delta_depth_close_2m_pp,2,delta_pitch_rad,IMU e atitude,11.62


**Maiores erros de validacao em `delta_depth_close_5m_pp` e suas tres entradas mais influentes:**

,run_id,sample_id,real,predito,erro_absoluto,feature_1,importancia_1_pct,feature_2,importancia_2_pct,feature_3,importancia_3_pct
0,run_20260716_221545,10,-8.16,-3.61,4.55,gyro_y_integral_rad,11.44,delta_z_m,7.66,delta_pitch_rad,7.33
1,run_20260715_215803,24,4.80,0.30,4.49,delta_pitch_rad,14.94,gyro_y_integral_rad,14.77,delta_gyro_y_rad_s,5.08
2,run_20260716_221545,32,3.68,-0.39,4.07,gyro_y_integral_rad,13.92,delta_pitch_rad,13.04,delta_z_m,7.58
3,run_20260716_221545,17,-3.21,0.71,3.91,gyro_y_integral_rad,14.26,delta_pitch_rad,13.31,delta_gyro_y_rad_s,5.39
4,run_20260716_221545,19,3.38,-0.20,3.58,gyro_y_integral_rad,11.54,delta_pitch_rad,10.95,delta_z_m,6.83
5,run_20260716_221545,30,-3.10,0.36,3.47,gyro_y_integral_rad,13.57,delta_pitch_rad,11.20,delta_gyro_y_rad_s,7.92
6,run_20260715_215803,13,-2.67,0.73,3.41,gyro_y_integral_rad,15.16,delta_pitch_rad,13.22,delta_gyro_y_rad_s,7.17
7,run_20260715_215803,15,-3.61,-0.38,3.23,gyro_y_integral_rad,11.31,delta_pitch_rad,10.12,delta_z_m,7.41
8,run_20260716_221545,27,4.42,1.22,3.20,gyro_y_integral_rad,11.95,delta_pitch_rad,11.93,delta_z_m,7.84
9,run_20260715_215803,12,-3.00,-0.16,2.84,delta_pitch_rad,13.26,gyro_y_integral_rad,12.26,delta_z_m,5.79


### Modelo em duas etapas: evento de proximidade + regressao nos eventos

Este experimento preserva a MLP multi-output anterior e adiciona uma avaliacao alternativa para o alvo `delta_depth_close_5m_pp`. A ideia e separar a pergunta em duas partes: primeiro detectar se houve uma mudanca relevante de proximidade e, so depois, estimar a magnitude/sinal dessa mudanca.

In [10]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import average_precision_score, balanced_accuracy_score, confusion_matrix, precision_recall_fscore_support
from sklearn.neural_network import MLPClassifier

EVENT_TARGET_DELTA = "delta_depth_close_5m_pp"
EVENT_THRESHOLD_PP = 0.5
EVENT_PROBA_THRESHOLD = 0.35
EVENT_TEMPORAL_LAGS = (1, 2, 3)


def signed_log1p_array(y, scale=EVENT_THRESHOLD_PP):
    y = np.asarray(y, dtype=float)
    return np.sign(y) * np.log1p(np.abs(y) / max(scale, 1e-6))


def signed_expm1_array(z, scale=EVENT_THRESHOLD_PP):
    z = np.asarray(z, dtype=float)
    return np.sign(z) * np.expm1(np.abs(z)) * max(scale, 1e-6)


def colunas_temporais_evento(X_base):
    """Seleciona as features adequadas para construir contexto temporal."""

    preferidas = [
        "dt_s", "depth_age_s", "depth_dt_s",
        "delta_x_m", "delta_y_m", "delta_z_m",
        "delta_roll_rad", "delta_pitch_rad", "delta_yaw_heading_rad",
        "flow_valid_points", "flow_track_retention_pct",
        "flow_mag_p90_px", "radial_flow_p90_px", "pan_comp_delta_rad",
        "img_delta_abs_mean", "img_delta_abs_p90",
        "flow_vec_radial_mean", "flow_vec_risk_mean", "flow_vec_rel_std",
    ]
    return [col for col in preferidas if col in X_base.columns]


def adicionar_contexto_temporal_evento(X_base, meta_base, lags=EVENT_TEMPORAL_LAGS):
    """Adiciona lags e tendencias sem misturar intervalos entre runs."""

    X_base = X_base.reset_index(drop=True).copy()
    meta_ordem = meta_base.reset_index(drop=True).copy()
    if "ordem_intervalo" not in meta_ordem.columns:
        meta_ordem["ordem_intervalo"] = np.arange(len(meta_ordem))

    colunas = colunas_temporais_evento(X_base)
    if not colunas:
        return X_base

    trabalho = pd.concat([
        meta_ordem[["run_id", "ordem_intervalo"]].reset_index(drop=True),
        X_base[colunas].reset_index(drop=True),
    ], axis=1)
    trabalho["orig_idx"] = np.arange(len(trabalho))
    trabalho = trabalho.sort_values(["run_id", "ordem_intervalo", "orig_idx"])

    novas_partes = [trabalho]
    for lag in lags:
        lagged = trabalho.groupby("run_id")[colunas].shift(lag)
        lagged.columns = [f"{col}_lag{lag}" for col in colunas]
        novas_partes.append(lagged)

    contexto = pd.concat(novas_partes, axis=1)
    for col in colunas:
        lag3 = f"{col}_lag3"
        if lag3 in contexto.columns:
            contexto[f"{col}_trend3"] = contexto[col] - contexto[lag3]

    contexto = contexto.sort_values("orig_idx")
    contexto = contexto.drop(columns=["run_id", "ordem_intervalo", "orig_idx"], errors="ignore")
    contexto = contexto.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return pd.concat([X_base, contexto.drop(columns=colunas, errors="ignore")], axis=1)


def preparar_dataset_evento_proximidade(X_base, y_base, meta_base, target=EVENT_TARGET_DELTA, threshold=EVENT_THRESHOLD_PP):
    """Cria o alvo binario de proximidade e seu contexto temporal."""

    if target not in y_base.columns:
        raise ValueError(f"Alvo {target} nao encontrado em y_base.")

    X_evento = adicionar_contexto_temporal_evento(X_base, meta_base)
    y_delta = y_base[target].reset_index(drop=True).astype(float)
    y_evento = (np.abs(y_delta) >= threshold).astype(int)
    meta_evento = meta_base.reset_index(drop=True).copy()
    meta_evento["delta_target"] = y_delta
    meta_evento["evento_proximidade"] = y_evento
    return X_evento, y_delta, y_evento, meta_evento


def criar_classificador_evento():
    """Cria o pipeline padrao do classificador de proximidade."""

    return Pipeline([
        ("x_scaler", StandardScaler()),
        ("mlp_evento", MLPClassifier(
            hidden_layer_sizes=(32, 16),
            activation="relu",
            solver="lbfgs",
            alpha=0.05,
            max_iter=2000,
            random_state=42,
        )),
    ])


def avaliar_classificacao_evento(y_true, y_pred, y_score, modelo, split):
    """Resume desempenho, matriz de confusao e average precision."""

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", zero_division=0
    )
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    if len(np.unique(y_true)) > 1 and np.std(y_score) > 1e-9:
        avg_precision = float(average_precision_score(y_true, y_score))
    else:
        avg_precision = np.nan

    return {
        "modelo": modelo,
        "split": split,
        "event_rate_real": float(np.mean(y_true)),
        "event_rate_pred": float(np.mean(y_pred)),
        "balanced_acc": float(balanced_accuracy_score(y_true, y_pred)),
        "precision_evento": float(precision),
        "recall_evento": float(recall),
        "f1_evento": float(f1),
        "avg_precision": avg_precision,
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }


def avaliar_delta_unico(y_true, y_pred, modelo, split, alvo=EVENT_TARGET_DELTA):
    """Avalia um unico alvo continuo apos o gate de evento."""

    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    corr = float(np.corrcoef(y_true, y_pred)[0, 1]) if len(y_true) > 1 and np.std(y_true) > 1e-9 and np.std(y_pred) > 1e-9 else np.nan
    return {
        "modelo": modelo,
        "split": split,
        "alvo_delta": alvo,
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "RMSE": float(mean_squared_error(y_true, y_pred) ** 0.5),
        "corr": corr,
    }



In [11]:
if "X_mlp" not in globals() or "y_mlp" not in globals() or "meta_mlp" not in globals():
    raiz_mlp = localizar_raiz_projeto_memmap()
    X_mlp, y_mlp, meta_mlp = montar_dataset_mlp_delta_depth(raiz_mlp)

if len(X_mlp) < 12 or EVENT_TARGET_DELTA not in y_mlp.columns:
    display(Markdown("Dataset insuficiente para o modelo em duas etapas de proximidade."))
else:
    runs_ordenadas_evento = list(meta_mlp["run_id"].drop_duplicates())
    resultados_classificacao_cumulativos = []
    for marco_runs in marcos_comparacao(len(runs_ordenadas_evento)):
        if len(runs_ordenadas_evento) < marco_runs:
            continue
        X_marco, y_marco, meta_marco = recortar_marco_runs(
            X_mlp, y_mlp, meta_mlp, runs_ordenadas_evento, marco_runs
        )
        X_evento_marco, _, y_evento_marco, meta_evento_marco = preparar_dataset_evento_proximidade(
            X_marco, y_marco, meta_marco
        )
        split_marco = separar_intervalos(meta_evento_marco)
        Xv_marco = X_evento_marco.replace([np.inf, -np.inf], np.nan).fillna(0.0).to_numpy(float)
        yv_marco = y_evento_marco.to_numpy(int)
        clf_marco = criar_classificador_evento()
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            clf_marco.fit(Xv_marco[split_marco["train"]], yv_marco[split_marco["train"]])
        for nome_split, idx in [("validacao", split_marco["val"]), ("teste", split_marco["test"])]:
            proba = clf_marco.predict_proba(Xv_marco[idx])[:, 1]
            pred = (proba >= EVENT_PROBA_THRESHOLD).astype(int)
            linha = avaliar_classificacao_evento(yv_marco[idx], pred, proba, "MLP evento", nome_split)
            linha["marco_runs"] = marco_runs
            linha["estrategia_split"] = "validacao_teste_fixos"
            resultados_classificacao_cumulativos.append(linha)

    resultados_classificacao_cumulativos = pd.DataFrame(resultados_classificacao_cumulativos)
    caminho_resultados_evento = raiz_mlp / "estudos_e_analises" / "comparacao_eventos_splits_fixos.csv"
    resultados_classificacao_cumulativos.to_csv(caminho_resultados_evento, index=False)
    display(Markdown("**Curva do classificador com validacao e teste fixos:**"))
    display(resultados_classificacao_cumulativos.sort_values(["split", "marco_runs"]).reset_index(drop=True))
    px.line(resultados_classificacao_cumulativos, x="marco_runs", y="balanced_acc", color="split", markers=True, title=f"Balanced accuracy com splits fixos: {sorted(resultados_classificacao_cumulativos['marco_runs'].unique())} runs", template="plotly_white").show()

    X_evento, y_delta_evento, y_evento, meta_evento = preparar_dataset_evento_proximidade(
        X_mlp, y_mlp, meta_mlp
    )
    split_evento = separar_intervalos(meta_evento)
    if len(split_evento["test"]) == 0:
        split_evento["test"] = split_evento["val"]
    if len(split_evento["val"]) == 0:
        split_evento["val"] = split_evento["test"]

    display(Markdown(
        f"**Dataset evento proximidade:** {len(X_evento)} intervalos, {X_evento.shape[1]} features "
        f"com contexto temporal, alvo `{EVENT_TARGET_DELTA}`, limiar={EVENT_THRESHOLD_PP:.2f} p.p. "
        f"Taxa de evento={float(y_evento.mean()):.1%}. Separacao: {split_evento['modo']}. "
        f"Treino={len(split_evento['train'])}, validacao={len(split_evento['val'])}, teste={len(split_evento['test'])}."
    ))

    resumo_runs_evento = meta_evento.groupby("run_id").agg(
        intervalos=("evento_proximidade", "size"),
        eventos=("evento_proximidade", "sum"),
        taxa_evento=("evento_proximidade", "mean"),
        delta_abs_p90=("delta_target", lambda s: float(np.percentile(np.abs(s), 90))),
    ).reset_index()
    display(resumo_runs_evento)

    Xv_evento = X_evento.replace([np.inf, -np.inf], np.nan).fillna(0.0).to_numpy(float)
    y_delta_v = y_delta_evento.to_numpy(float)
    y_evento_v = y_evento.to_numpy(int)

    train_idx = split_evento["train"]
    train_event_idx = train_idx[y_evento_v[train_idx] == 1]

    if len(np.unique(y_evento_v[train_idx])) < 2 or len(train_event_idx) < 5:
        display(Markdown(
            "A separacao atual nao tem exemplos suficientes de evento no treino para classificador + regressor. "
            "Diminua `EVENT_THRESHOLD_PP` ou colete mais runs com aproximacao real de obstaculos."
        ))
    else:
        clf_evento = criar_classificador_evento()
        clf_dummy = DummyClassifier(strategy="most_frequent")

        reg_evento = EarlyStoppingDropoutRegressor(**REGRESSOR_CONFIG)
        reg_dummy_evento = DummyRegressor(strategy="median")

        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            clf_evento.fit(Xv_evento[train_idx], y_evento_v[train_idx])
            val_event_idx = split_evento["val"][y_evento_v[split_evento["val"]] == 1]
            test_event_idx = split_evento["test"][y_evento_v[split_evento["test"]] == 1]
            reg_evento.fit(
                Xv_evento[train_event_idx],
                signed_log1p_array(y_delta_v[train_event_idx]),
                Xv_evento[val_event_idx],
                signed_log1p_array(y_delta_v[val_event_idx]),
                Xv_evento[test_event_idx],
                signed_log1p_array(y_delta_v[test_event_idx]),
            )

        clf_dummy.fit(Xv_evento[train_idx], y_evento_v[train_idx])
        reg_dummy_evento.fit(Xv_evento[train_event_idx], y_delta_v[train_event_idx])

        linhas_classificacao = []
        linhas_regressao = []
        predicoes_evento = {}

        for nome_split, idx in [("validacao", split_evento["val"]), ("teste", split_evento["test"] )]:
            if len(idx) == 0:
                continue

            proba_evento = clf_evento.predict_proba(Xv_evento[idx])[:, 1]
            pred_evento = (proba_evento >= EVENT_PROBA_THRESHOLD).astype(int)
            pred_evento_dummy = clf_dummy.predict(Xv_evento[idx]).astype(int)
            score_dummy = np.full(len(idx), float(np.mean(y_evento_v[train_idx])))

            linhas_classificacao.append(avaliar_classificacao_evento(
                y_evento_v[idx], pred_evento, proba_evento, "MLP evento", nome_split
            ))
            linhas_classificacao.append(avaliar_classificacao_evento(
                y_evento_v[idx], pred_evento_dummy, score_dummy, "Classe majoritaria", nome_split
            ))

            pred_delta_evento = signed_expm1_array(reg_evento.predict(Xv_evento[idx]).ravel())
            pred_delta_gate = np.where(pred_evento == 1, pred_delta_evento, 0.0)
            pred_delta_evento_real_gate = np.where(y_evento_v[idx] == 1, pred_delta_evento, 0.0)
            pred_delta_dummy_gate = np.where(pred_evento == 1, reg_dummy_evento.predict(Xv_evento[idx]), 0.0)
            pred_delta_zero = np.zeros(len(idx), dtype=float)

            linhas_regressao.extend([
                avaliar_delta_unico(y_delta_v[idx], pred_delta_gate, "Pipeline completo", nome_split),
                avaliar_delta_unico(y_delta_v[idx], pred_delta_evento_real_gate, "Evento real + regressor (diagnostico)", nome_split),
                avaliar_delta_unico(y_delta_v[idx], pred_delta_dummy_gate, "Classificador + mediana", nome_split),
                avaliar_delta_unico(y_delta_v[idx], pred_delta_zero, "Sempre prever zero", nome_split),
            ])

            predicoes_evento[nome_split] = {
                "idx": idx,
                "proba_evento": proba_evento,
                "evento_real": y_evento_v[idx],
                "evento_pred": pred_evento,
                "delta_real": y_delta_v[idx],
                "delta_pred_gate": pred_delta_gate,
                "delta_pred_evento_real_gate": pred_delta_evento_real_gate,
            }

        resultados_classificacao_evento = pd.DataFrame(linhas_classificacao)
        resultados_regressao_evento = pd.DataFrame(linhas_regressao)

        display(Markdown(
            "### Etapa 1 - detectar se houve mudanca relevante de proximidade\n"
            "A classe majoritaria preve evento em todos os intervalos. Por isso ela tem recall 1, "
            "mas balanced accuracy 0,5: encontrar todos os eventos nao significa separar bem as duas classes."
        ))
        display(resultados_classificacao_evento.round(3))

        metricas_classificador = (
            resultados_classificacao_evento
            .query("modelo == 'MLP evento'")
            .melt(
                id_vars=["split"],
                value_vars=["balanced_acc", "precision_evento", "recall_evento", "f1_evento"],
                var_name="metrica",
                value_name="valor",
            )
        )
        metricas_classificador["metrica"] = metricas_classificador["metrica"].map({
            "balanced_acc": "Balanced accuracy",
            "precision_evento": "Precisao",
            "recall_evento": "Recall",
            "f1_evento": "F1",
        })
        fig_metricas_evento = px.bar(
            metricas_classificador,
            x="metrica",
            y="valor",
            color="split",
            barmode="group",
            text="valor",
            range_y=[0, 1.05],
            title="Desempenho do classificador de eventos",
            labels={"metrica": "Metrica", "valor": "Resultado", "split": "Conjunto"},
            template="plotly_white",
        )
        fig_metricas_evento.update_traces(texttemplate="%{text:.3f}", textposition="outside")
        fig_metricas_evento.show()

        resultado_clf_teste = resultados_classificacao_evento.query(
            "modelo == 'MLP evento' and split == 'teste'"
        ).iloc[0]
        matriz_teste = np.array([
            [resultado_clf_teste["tn"], resultado_clf_teste["fp"]],
            [resultado_clf_teste["fn"], resultado_clf_teste["tp"]],
        ], dtype=int)
        fig_confusao = go.Figure(go.Heatmap(
            z=matriz_teste,
            x=["Predito: sem evento", "Predito: evento"],
            y=["Real: sem evento", "Real: evento"],
            text=matriz_teste,
            texttemplate="%{text}",
            colorscale="Blues",
            showscale=False,
        ))
        fig_confusao.update_layout(
            title="Onde o classificador acertou e errou no teste fixo",
            template="plotly_white",
            yaxis_autorange="reversed",
        )
        fig_confusao.show()

        display(Markdown(
            "### Etapa 2 - estimar o tamanho da mudanca\n"
            f"O regressor de magnitude usa dropout de {reg_evento.dropout:.0%} e early stopping. "
            f"A melhor epoca foi {reg_evento.best_epoch_}, com interrupcao na epoca "
            f"{reg_evento.stopped_epoch_}.\n\n"
            "`Evento real + regressor (diagnostico)` informa ao regressor quais intervalos realmente "
            "tinham evento. Essa opcao nao pode ser usada em producao; ela serve apenas para verificar "
            "se o erro vem principalmente do classificador ou do regressor."
        ))
        resultados_regressao_teste = (
            resultados_regressao_evento
            .query("split == 'teste'")
            .sort_values("MAE")
            .reset_index(drop=True)
        )
        display(resultados_regressao_teste.round(3))
        fig_mae_teste = px.bar(
            resultados_regressao_teste,
            x="modelo",
            y="MAE",
            color="modelo",
            text="MAE",
            title=f"Erro final no teste fixo para {EVENT_TARGET_DELTA}",
            labels={"modelo": "Estrategia", "MAE": "MAE (p.p.)"},
            template="plotly_white",
        )
        fig_mae_teste.update_traces(texttemplate="%{text:.3f}", textposition="outside")
        fig_mae_teste.update_layout(showlegend=False)
        fig_mae_teste.show()

        mae_pipeline = resultados_regressao_teste.loc[
            resultados_regressao_teste["modelo"] == "Pipeline completo", "MAE"
        ].iloc[0]
        mae_evento_real = resultados_regressao_teste.loc[
            resultados_regressao_teste["modelo"] == "Evento real + regressor (diagnostico)", "MAE"
        ].iloc[0]
        variacao_evento_real_pct = (mae_evento_real / max(mae_pipeline, 1e-9) - 1.0) * 100.0
        if abs(variacao_evento_real_pct) <= 5.0:
            leitura_diagnostico = (
                "Os valores ficaram proximos, entao os erros do classificador nao explicam a maior "
                "parte do erro final."
            )
        elif variacao_evento_real_pct < 0.0:
            leitura_diagnostico = (
                "O MAE diminuiu com o evento real, indicando que os erros do classificador contribuem "
                "para o erro final."
            )
        else:
            leitura_diagnostico = (
                "O MAE aumentou mesmo com o evento real. Isso nao indica ganho do classificador; mostra "
                "que o regressor de magnitude ainda esta instavel nos intervalos de evento."
            )
        display(Markdown(
            f"**Leitura direta:** no teste, fornecer o evento real ao regressor mudou o MAE de "
            f"{mae_pipeline:.3f} para {mae_evento_real:.3f} p.p. {leitura_diagnostico}"
        ))

        maior_mae_validacao = resultados_regressao_evento.query("split == 'validacao'")["MAE"].max()
        maior_mae_teste = resultados_regressao_teste["MAE"].max()
        if maior_mae_validacao > max(100.0, maior_mae_teste * 10.0):
            display(Markdown(
                f"**Alerta de instabilidade:** o maior MAE da validacao foi {maior_mae_validacao:.3e} p.p. "
                "Esse valor extremo achatava as demais barras no grafico anterior. O grafico principal acima "
                "mostra somente o teste fixo, mas a divergencia da validacao continua registrada e indica "
                "que o regressor desta segunda etapa ainda nao esta estavel."
            ))

        if "teste" in predicoes_evento:
            pred_teste_evento = predicoes_evento["teste"]
            meta_teste_evento = meta_evento.iloc[pred_teste_evento["idx"]].reset_index(drop=True)
            inspecao_evento = pd.DataFrame({
                "run_id": meta_teste_evento["run_id"],
                "sample_id": meta_teste_evento["sample_id"],
                "delta_real": pred_teste_evento["delta_real"],
                "evento_real": pred_teste_evento["evento_real"],
                "proba_evento": pred_teste_evento["proba_evento"],
                "evento_pred": pred_teste_evento["evento_pred"],
                "delta_pred_gate": pred_teste_evento["delta_pred_gate"],
                "delta_pred_evento_real_gate": pred_teste_evento["delta_pred_evento_real_gate"],
            })
            display(Markdown("**Intervalos de teste mais extremos para inspecao:**"))
            display(inspecao_evento.assign(abs_delta=lambda df: df["delta_real"].abs()).sort_values("abs_delta", ascending=False).drop(columns="abs_delta").head(20))



**Curva do classificador com validacao e teste fixos:**

,modelo,split,event_rate_real,event_rate_pred,balanced_acc,precision_evento,recall_evento,f1_evento,avg_precision,tn,fp,fn,tp,marco_runs,estrategia_split
0,MLP evento,teste,0.734043,0.851064,0.562029,0.762500,0.884058,0.818792,0.850189,6,19,8,61,9,validacao_teste_fixos
1,MLP evento,teste,0.734043,0.776596,0.565797,0.767123,0.811594,0.788732,0.809643,8,17,13,56,17,validacao_teste_fixos
2,MLP evento,teste,0.734043,0.797872,0.689275,0.826667,0.898551,0.861111,0.866608,12,13,7,62,25,validacao_teste_fixos
3,MLP evento,teste,0.734043,0.744681,0.544058,0.757143,0.768116,0.762590,0.880318,8,17,16,53,33,validacao_teste_fixos
4,MLP evento,teste,0.734043,0.808511,0.614783,0.789474,0.869565,0.827586,0.873330,9,16,9,60,40,validacao_teste_fixos
5,MLP evento,validacao,0.764706,0.729412,0.617308,0.822581,0.784615,0.803150,0.868601,9,11,14,51,9,validacao_teste_fixos
6,MLP evento,validacao,0.764706,0.729412,0.617308,0.822581,0.784615,0.803150,0.881216,9,11,14,51,17,validacao_teste_fixos
7,MLP evento,validacao,0.764706,0.635294,0.719231,0.888889,0.738462,0.806723,0.887119,14,6,17,48,25,validacao_teste_fixos
8,MLP evento,validacao,0.764706,0.694118,0.626923,0.830508,0.753846,0.790323,0.873080,10,10,16,49,33,validacao_teste_fixos
9,MLP evento,validacao,0.764706,0.752941,0.665385,0.843750,0.830769,0.837209,0.851854,10,10,11,54,40,validacao_teste_fixos


**Dataset evento proximidade:** 1234 intervalos, 120 features com contexto temporal, alvo `delta_depth_close_5m_pp`, limiar=0.50 p.p. Taxa de evento=69.5%. Separacao: por run_id com validacao/teste fixos. Treino=1055, validacao=85, teste=94.

,run_id,intervalos,eventos,taxa_evento,delta_abs_p90
0,run_20260715_163428,33,22,0.666667,2.758374
1,run_20260715_163825,43,29,0.674419,3.663851
2,run_20260715_215803,46,38,0.826087,3.518113
3,run_20260715_220110,36,24,0.666667,2.562000
4,run_20260715_220346,30,20,0.666667,4.174669
5,run_20260715_220437,32,21,0.656250,3.797189
6,run_20260716_221426,32,26,0.812500,3.435565
7,run_20260716_221545,39,27,0.692308,3.442211
8,run_20260716_231519,32,28,0.875000,4.051497
9,run_20260721_102934,38,28,0.736842,5.485666


### Etapa 1 - detectar se houve mudanca relevante de proximidade
A classe majoritaria preve evento em todos os intervalos. Por isso ela tem recall 1, mas balanced accuracy 0,5: encontrar todos os eventos nao significa separar bem as duas classes.

...

### Etapa 2 - estimar o tamanho da mudanca
O regressor de magnitude usa dropout de 30% e early stopping. A melhor epoca foi 9, com interrupcao na epoca 59.

`Evento real + regressor (diagnostico)` informa ao regressor quais intervalos realmente tinham evento. Essa opcao nao pode ser usada em producao; ela serve apenas para verificar se o erro vem principalmente do classificador ou do regressor.

,modelo,split,alvo_delta,MAE,RMSE,corr
0,Evento real + regressor (diagnostico),teste,delta_depth_close_5m_pp,1.744,2.810,0.250
1,Pipeline completo,teste,delta_depth_close_5m_pp,1.770,2.810,0.248
2,Sempre prever zero,teste,delta_depth_close_5m_pp,1.834,2.874,NaN
3,Classificador + mediana,teste,delta_depth_close_5m_pp,1.919,2.933,0.110


**Leitura direta:** no teste, fornecer o evento real ao regressor mudou o MAE de 1.770 para 1.744 p.p. Os valores ficaram proximos, entao os erros do classificador nao explicam a maior parte do erro final.

**Intervalos de teste mais extremos para inspecao:**

,run_id,sample_id,delta_real,evento_real,proba_evento,evento_pred,delta_pred_gate,delta_pred_evento_real_gate
18,run_20260715_220346,20,-14.438908,1,1.000000,1,-0.282936,-0.282936
11,run_20260715_220346,12,8.220404,1,0.110655,0,0.000000,-0.210111
53,run_20260715_220437,28,8.163571,1,0.999923,1,0.558335,0.558335
17,run_20260715_220346,19,-7.065944,1,1.000000,1,-0.048727,-0.048727
44,run_20260715_220437,17,6.492569,1,0.999994,1,0.617056,0.617056
54,run_20260715_220437,29,5.777230,1,1.000000,1,0.050599,0.050599
75,run_20260716_231519,14,-5.210536,1,1.000000,1,-0.025257,-0.025257
93,run_20260716_231519,34,-5.096783,1,0.999940,1,-0.348774,-0.348774
77,run_20260716_231519,16,-4.357495,1,0.999800,1,0.004537,0.004537
76,run_20260716_231519,15,4.100497,1,1.000000,1,0.177994,0.177994


### Validacao da sincronizacao entre flow, IMU e depth

A pergunta principal Ã© se os intervalos salvos em memmap estao coerentes: `dt_s`, `depth_age_s`, `depth_dt_s`, vetores de optical flow e deltas de depth devem descrever a mesma janela temporal da logica de proximidade visual.

In [12]:
COLUNAS_SYNC_INTERVALOS = ["dt_s", "depth_age_s", "depth_dt_s", "flow_valid_points", "flow_track_retention_pct", "flow_mag_p90_px", "radial_flow_p90_px", "delta_depth_close_5m_pp", "delta_x_m", "delta_y_m", "delta_yaw_heading_rad", "pan_comp_delta_rad"]


def montar_dataframe_intervalos_memmap(base_dir):
    """Concatena os intervalos sincronizados de todas as runs depth."""

    runs = carregar_todas_runs_depth_memmap(base_dir)
    if not runs:
        return pd.DataFrame()
    partes = []
    for run in runs:
        df = run["intervalos"].copy()
        df["run_id"] = run["run_dir"].name
        partes.append(df)
    return pd.concat(partes, ignore_index=True) if partes else pd.DataFrame()


def resumir_sincronizacao_intervalos(df):
    if df.empty:
        return pd.DataFrame()
    cols = [c for c in COLUNAS_SYNC_INTERVALOS if c in df.columns]
    return df[cols].describe().T


raiz_flow = localizar_raiz_projeto_memmap()
intervalos_flow_df = montar_dataframe_intervalos_memmap(raiz_flow)

if intervalos_flow_df.empty:
    display(Markdown("Nenhum intervalo memmap encontrado para validar sincronizacao flow/depth."))
else:
    print(f"Intervalos avaliados: {len(intervalos_flow_df)}")
    print(intervalos_flow_df.groupby("run_id").size().rename("intervalos"))
    display(resumir_sincronizacao_intervalos(intervalos_flow_df))
    px.box(intervalos_flow_df, x="run_id", y="dt_s", title="Distribuicao do intervalo temporal entre atualizacoes visuais", labels={"dt_s": "Delta de tempo do intervalo visual (s)", "run_id": "Run"}, template="plotly_white").show()
    px.scatter(intervalos_flow_df, x="radial_flow_p90_px", y="delta_depth_close_5m_pp", color="run_id", size="flow_valid_points", hover_data=["sample_id", "dt_s", "depth_age_s", "pan_comp_delta_rad"], title="Flow radial do intervalo x variacao de proximidade no depth", labels={"radial_flow_p90_px": "P90 do flow radial (px)", "delta_depth_close_5m_pp": "Delta pixels < 5 m (p.p.)"}, template="plotly_white").show()
    serie = intervalos_flow_df.reset_index(drop=True)
    fig_tempo = go.Figure()
    fig_tempo.add_trace(go.Scatter(x=serie.index, y=serie["flow_mag_p90_px"], mode="lines", name="P90 flow"))
    fig_tempo.add_trace(go.Scatter(x=serie.index, y=serie["delta_depth_close_5m_pp"], mode="lines", name="Delta pixels < 5m", yaxis="y2"))
    fig_tempo.update_layout(title="Sequencia dos intervalos: flow visual e delta de proximidade", xaxis_title="Intervalos concatenados em ordem de leitura", yaxis=dict(title="P90 flow (px)"), yaxis2=dict(title="Delta pixels < 5m (p.p.)", overlaying="y", side="right"), template="plotly_white", hovermode="x unified")
    fig_tempo.show()

Intervalos avaliados: 1234
run_id
run_20260715_163428    33
run_20260715_163825    43
run_20260715_215803    46
run_20260715_220110    36
run_20260715_220346    30
run_20260715_220437    32
run_20260716_221426    32
run_20260716_221545    39
run_20260716_231519    32
run_20260721_102934    38
run_20260721_111722    44
run_20260721_112611    50
run_20260721_113351    56
run_20260721_121851    42
run_20260721_123310    53
run_20260721_124311    61
run_20260721_130115    56
run_20260723_111141    13
run_20260723_114842    11
run_20260723_115645    18
run_20260723_122007    13
run_20260723_122339     6
run_20260723_123404    16
run_20260723_123607    11
run_20260723_123848    16
run_20260723_125240    17
run_20260723_125508    12
run_20260723_125716     8
run_20260723_130416     9
run_20260723_130458    14
run_20260723_140857    39
run_20260723_143243    43
run_20260723_143604    46
run_20260723_143959    49
run_20260723_144135    31
run_20260723_155203    46
run_20260723_160928    26
run_

,count,mean,std,min,25%,50%,75%,max
dt_s,1234.0,0.106311,0.054188,0.032000,0.068000,0.100000,0.132000,0.460000
depth_age_s,1234.0,0.045167,0.016037,0.032000,0.032000,0.036000,0.064000,0.068000
depth_dt_s,1234.0,0.119154,0.063292,0.032000,0.068000,0.100000,0.164000,0.464000
flow_valid_points,1234.0,81.444895,12.280803,20.000000,76.000000,82.000000,89.000000,134.000000
flow_track_retention_pct,1234.0,97.119873,5.584471,54.444443,96.725767,100.000000,100.000000,100.000000
flow_mag_p90_px,1234.0,38.943497,44.946930,0.000000,9.261493,22.828687,51.618120,316.002319
radial_flow_p90_px,1234.0,32.486931,36.671356,-0.003340,7.842049,19.729582,44.384052,256.597290
delta_depth_close_5m_pp,1234.0,-0.085534,3.108788,-30.047611,-1.264367,-0.000115,1.066321,20.498875
delta_x_m,1212.0,-0.010260,0.453057,-2.187938,-0.263824,-0.001320,0.226845,2.006323
delta_y_m,1212.0,0.013577,0.633620,-3.247337,-0.434906,0.002575,0.419971,3.277081
